In [ ]:
import pandas as pd
import re
import time
import numpy as np
from itertools import combinations


# Load datasets
acm = pd.read_csv("ACM.csv")
dblp = pd.read_csv("DBLP2.csv", encoding="latin1") # breaks without this
perfect_mapping = pd.read_csv("DBLP-ACM_perfectMapping.csv")

# (a) Concat
def concatenate_record(rec):
    return " ".join(str(val) for val in rec.values)
    
def preprocess(text):
    # (b) lowercase
    text = text.lower()
    # (c) convert multiple spaces to one
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
    

# Preprocess 
for df in [acm, dblp]:
    for col in ['title', 'authors', 'venue']:
        if col != "id":
            df[col] = df[col].astype(str).apply(preprocess)

# Keep IDs separately
acm_ids = acm["id"].tolist()
dblp_ids = dblp["id"].tolist()


# Convert each row into one concatenated string
acm_strings = acm.drop('id', axis=1).apply(concatenate_record, axis=1).tolist()
dblp_strings = dblp.drop('id', axis=1).apply(concatenate_record, axis=1).tolist()

doc_ids = acm_ids + dblp_ids
doc_strings = acm_strings + dblp_strings



In [ ]:
# This whole cell is lab code, some descriptions are removed from the functions to make it shorter
def get_minhash_arr(num_hashes:int,vocab:dict):
    """
    """
    length = len(vocab.keys())
    arr = np.zeros((num_hashes,length))
    for i in range(num_hashes):
        permutation = np.random.permutation(len(vocab.keys()))
        arr[i,:] = permutation.copy()
    return arr.astype(int)
def get_signature(minhash:np.ndarray, vector:np.ndarray):
    """
    """
    idx = np.nonzero(vector)[0].tolist()
    shingles = minhash[:,idx]
    signature = np.min(shingles,axis=1)
    return signature

def jaccard_similarity(set1, set2):
    intersection_size = len(set1.intersection(set2))
    union_size = len(set1.union(set2))
    return intersection_size / union_size if union_size != 0 else 0.0

def compute_signature_similarity(signature_1, signature_2):
    """
    """
    # Ensure the matrices have the same shape
    if signature_1.shape != signature_2.shape:
        raise ValueError("Both signature matrices must have the same shape.")
    # Count the number of rows where the two matrices agree
    agreement_count = np.sum(signature_1 == signature_2)
    # Calculate the similarity
    similarity = agreement_count / signature_2.shape[0]

    return similarity

class LSH:
    """
    Implements the Locality Sensitive Hashing (LSH) technique for approximate
    nearest neighbor search.
    """
    buckets = []
    counter = 0

    def __init__(self, b: int):
        """
        Initializes the LSH instance with a specified number of bands.

        Parameters:
        - b (int): The number of bands to divide the signature into.
        """
        self.b = b
        for i in range(b):
            self.buckets.append({})

    def make_subvecs(self, signature: np.ndarray) -> np.ndarray:
        """
        Divides a given signature into subvectors based on the number of bands.

        Parameters:
        - signature (np.ndarray): The MinHash signature to be divided.

        Returns:
        - np.ndarray: A stacked array where each row is a subvector of the signature.
        """
        l = len(signature)
        assert l % self.b == 0
        r = int(l / self.b)
        subvecs = []
        for i in range(0, l, r):
            subvecs.append(signature[i:i+r])
        return np.stack(subvecs)

    def add_hash(self, signature: np.ndarray):
        """
        Adds a signature to the appropriate LSH buckets based on its subvectors.

        Parameters:
        - signature (np.ndarray): The MinHash signature to be hashed and added.
        """
        subvecs = self.make_subvecs(signature).astype(str)
        for i, subvec in enumerate(subvecs):
            subvec = ','.join(subvec)
            if subvec not in self.buckets[i].keys():
                self.buckets[i][subvec] = []
            self.buckets[i][subvec].append(self.counter)
        self.counter += 1

    def check_candidates(self) -> set:
        """
        Identifies candidate pairs from the LSH buckets that could be potential near duplicates.

        Returns:
        - set: A set of tuple pairs representing the indices of candidate signatures.
        """
        candidates = []
        for bucket_band in self.buckets:
            keys = bucket_band.keys()
            for bucket in keys:
                hits = bucket_band[bucket]
                if len(hits) > 1:
                    candidates.extend(combinations(hits, 2))
        return set(candidates)


def shingle(text: str, k: int)->set:
    """
    Create a set of 'shingles' from the input text using k-shingling.

    Parameters:
        text (str): The input text to be converted into shingles.
        k (int): The length of the shingles (substring size).

    Returns:
        set: A set containing the shingles extracted from the input text.
    """
    shingle_set = []
    for i in range(len(text) - k+1):
        shingle_set.append(text[i:i+k])
    return set(shingle_set)
def build_vocab(shingle_sets: list)->dict:
    """
    Constructs a vocabulary dictionary from a list of shingle sets.

    This function takes a list of shingle sets and creates a unified vocabulary
    dictionary. Each unique shingle across all sets is assigned a unique integer
    identifier.

    Parameters:
    - shingle_sets (list of set): A list containing sets of shingles.

    Returns:
    - dict: A vocabulary dictionary where keys are the unique shingles and values
      are their corresponding unique integer identifiers.

    Example:
    sets = [{"apple", "banana"}, {"banana", "cherry"}]
    build_vocab(sets)
    {'apple': 0, 'cherry': 1, 'banana': 2}  # The exact order might vary due to set behavior
    """
    full_set = {item for set_ in shingle_sets for item in set_}
    vocab = {}
    for i, shingle in enumerate(list(full_set)):
        vocab[shingle] = i
    return vocab
def one_hot(shingles: set, vocab: dict):
    vec = np.zeros(len(vocab))
    for shingle in shingles:
        idx = vocab[shingle]
        vec[idx] = 1
    return vec




In [ ]:
# Shingling

#### lab code

k = 3
shingles = []
for sentence in doc_strings:
    shingles.append(shingle(sentence,k))

# Build the vocab
vocab = build_vocab(shingles)
# 1hot
shingles_1hot = []
for shingle_set in shingles:
    shingles_1hot.append(one_hot(shingle_set,vocab))
shingles_1hot = np.stack(shingles_1hot)

# Min hash and signatures
minhash_arr =  get_minhash_arr(140,vocab)
signatures = []
for vector in shingles_1hot:
    signatures.append(get_signature(minhash_arr,vector))
signatures = np.stack(signatures)

# Build the candidate pairs with LSH
start = time.time()
b = 20   # number of buckets
lsh = LSH(b)
for signature in signatures:
    lsh.add_hash(signature)
candidate_pairs = lsh.check_candidates()

##### end of lab code

candidates = []
for i, j in candidate_pairs:


    # This piece of code makes sure we only compare records from the different two data sets since we made them all go into one list.
    # It makes sure the index is never in the same data set by using the length of the acm index as a sort of border
    # if the first one is in the range of the acm data set the second one shouldnt be and vice versa
    sim = compute_signature_similarity(signatures[i], signatures[j])
    id1, id2 = doc_ids[i], doc_ids[j]
    if i < len(acm_ids) and j >= len(acm_ids):
        candidates.append((id1, id2, sim))
    elif j < len(acm_ids) and i >= len(acm_ids):
        candidates.append((id2, id1, sim))

# Sort the candidates on score giving us top N candidates
candidates.sort(key=lambda x: x[2], reverse=True)

N = 2224
top_candidates = candidates[:N]

# This makes pairs of the perfect mappings and compares it to our pairs.
perfect_set = set(zip(perfect_mapping["idACM"], perfect_mapping["idDBLP"]))
reported_set = set((id1, id2) for id1, id2, _ in candidates)

correct_matches = reported_set & perfect_set
precision = len(correct_matches) / len(reported_set) if reported_set else 0
end = time.time()
runtime = end - start

print(f"Top-{N} LSH Candidate Pairs: {len(reported_set)}")
print(f"Correct Matches: {len(correct_matches)}")
print(f"LSH Precision: {precision:.4f}")
print(f"Running time: {runtime:.2f} seconds")


2672
Top-2224 LSH Candidate Pairs: 1859
Correct Matches: 1640
LSH Precision: 0.8822
Running time: 0.46 seconds
